# 🚗 AI-Based Driver Behavior Analysis Using Smartphone Sensors & Computer Vision
### Complete End-to-End Google Colab Workflow: From Raw Telemetry to Multimodal Risk Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjayGaikwad-07/driver-behavior-ml/blob/main/driver_behavior_colab_master.ipynb)

---
### 📌 Project Overview
This project develops an intelligent driver monitoring and behavior analysis system combining **smartphone inertial sensor telemetry** (accelerometer, gyroscope, speed, g-force) with **edge computer vision** (face landmark analysis for PERCLOS/drowsiness, and YOLO object detection for distracted driving habits like phone usage and smoking).

### 🗺️ Notebook Architecture & 5-Day Roadmap:
1. **Environment Setup & Requirements**: Verify GPU, install modern CV & ML dependencies.
2. **Data Ingestion & Path Configuration**: Mount Google Drive or auto-clone repo with balanced sampling.
3. **Day 2: Exploratory Data Analysis (EDA)**: Inspect label distributions, assess severe class imbalance, audit missing sensor telemetry.
4. **Day 3: Driver-Separated Group Splitting**: Zero-leakage data partition by `userid` to test generalization on unseen drivers.
5. **Day 3: Baseline Classifier & Fair Evaluation**: Random Forest vs. Dummy baseline, macro-F1, balanced accuracy, and confusion matrix.
6. **Day 4: Feature Ablation Study**: Comparative benchmark of Motion Features vs. State Proxies vs. Combined Sensors across Random Forest and ExtraTrees.
7. **Day 5: Computer Vision Pipeline**:
   - MediaPipe Face Landmarker for Eye Aspect Ratio (EAR) & Mouth Aspect Ratio (MAR) fatigue detection.
   - YOLO open-vocabulary / object detection for distraction identification (cell phone, smoking).
8. **Day 5: Multimodal Journey Risk Scoring**: Synthesize vision events and kinematic telemetry into a trip safety index (0–100) and exportable safety report.
9. **Interactive Streamlit Web Dashboard**: Launch the monitoring app directly from Colab via LocalTunnel.


---
## Step 1: Requirements & Hardware Check

In Google Colab, select: **Runtime > Change runtime type > T4 GPU** (or CPU if you only need the tabular ML modules).
The cell below checks available GPU hardware and installs all required dependencies.


In [ ]:
# Check GPU availability
import os
import sys
import torch

print("Python version:", sys.version)
if torch.cuda.is_available():
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("ℹ️ Running on CPU runtime. GPU is optional for tabular models, but recommended for YOLO/video inference.")


### Install Dependencies
We install `ultralytics` for YOLO, `mediapipe` for Face Mesh landmarking, `scikit-learn`, `seaborn`, `joblib`, and `streamlit`.


In [ ]:
# Install required libraries silently
!pip install -q --upgrade pip
!pip install -q ultralytics mediapipe opencv-python-headless scikit-learn pandas numpy matplotlib seaborn joblib streamlit
print("✅ All core libraries installed successfully!")


---
## Step 2: Repository Setup & Data Path Configuration

You can either:
- **Option A (Recommended)**: Auto-clone the GitHub repository into the Colab environment.
- **Option B**: Mount your personal Google Drive if you stored the full 9 GB `DriverSVT.csv` or customized models.


In [ ]:
from pathlib import Path
import os
import urllib.request

# Clone repository if running in fresh Google Colab environment
REPO_URL = "https://github.com/AjayGaikwad-07/driver-behavior-ml.git"
REPO_DIR = Path("driver-behavior-ml")

if not REPO_DIR.exists() and not Path("day3_prepare_dataset.py").exists():
    print(f"Cloning project repository from {REPO_URL} ...")
    !git clone $REPO_URL
    os.chdir(REPO_DIR)
    print("Changed directory to:", os.getcwd())
else:
    print("Project directory already active:", os.getcwd())

# Create necessary directories
for folder in ["data/demo", "data/raw", "models", "reports"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Ensure MediaPipe Face Landmarker model asset is downloaded
TASK_PATH = Path("models/face_landmarker.task")
if not TASK_PATH.exists():
    print("Downloading MediaPipe face_landmarker.task model...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, TASK_PATH)
    print(f"✅ Downloaded face_landmarker.task ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")
else:
    print(f"✅ MediaPipe face_landmarker.task found ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")


---
## Step 3: Dataset Ingestion

The full **DriverSVT** dataset contains ~9 GB of raw telemetry data.
For agile Colab experimentation, we use the **driver-separated balanced sample** (`day3_balanced_group_split_fast/sampled_driver_data_balanced_group_split.csv` ~68 MB), containing 580,806 rows from 491 drivers with all 9 behavioral target classes represented.


In [ ]:
import pandas as pd
import numpy as np

# Primary path: Balanced driver-separated sample
DATASET_PATH = Path("day3_balanced_group_split_fast/sampled_driver_data_balanced_group_split.csv")

# Fallback paths
if not DATASET_PATH.exists():
    alt_paths = [
        Path("day3_working/sampled_driver_data.csv"),
        Path("data/raw/DriverSVT/DriverSVT.csv")
    ]
    for p in alt_paths:
        if p.exists():
            DATASET_PATH = p
            break

print(f"Loading dataset from: {DATASET_PATH} ...")
# Read data with optimized memory
df = pd.read_csv(DATASET_PATH, low_memory=False)

print(f"✅ Loaded successfully! Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique drivers (userid): {df['userid'].nunique():,}")
df.head()


---
## Step 4: Exploratory Data Analysis (EDA) & Data Quality Audit

Let's inspect:
1. **Target distribution & Class Imbalance**: Most driving behavior datasets have overwhelming representation of "Normal" driving and very few hazardous instances ("Smoking", "Using phone", "Drowsiness").
2. **Missing value percentages** across inertial sensors and facial proxies.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Target Label Distribution
target_col = "dangerousstate"
class_counts = df[target_col].value_counts()
class_pcts = df[target_col].value_counts(normalize=True) * 100

colors = sns.color_palette("viridis", len(class_counts))
axes[0].barh(class_counts.index, class_counts.values, color=colors)
axes[0].set_title(f"Target Distribution: '{target_col}' (Total: {len(df):,})", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Sample Count")
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pcts.values)):
    axes[0].text(cnt + max(class_counts.values)*0.01, i, f"{cnt:,} ({pct:.1f}%)", va="center", fontsize=9)

# 2. Missing Value Audit
features_to_check = [
    "speed", "gforce", "acc_X", "acc_Y", "acc_Z",
    "accelerometer_data_X", "accelerometer_data_Y", "accelerometer_data_Z",
    "gyroscope_data_X", "gyroscope_data_Y", "gyroscope_data_Z",
    "perclos", "mouth_open_ratio"
]
missing_rates = df[features_to_check].isnull().mean() * 100

axes[1].barh(missing_rates.index, missing_rates.values, color="coral")
axes[1].set_title("Missing Value Frequency (%) by Feature", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Missing Rate (%)")
axes[1].set_xlim(0, max(max(missing_rates.values) * 1.5, 2.0))
for i, rate in enumerate(missing_rates.values):
    axes[1].text(rate + 0.05, i, f"{rate:.3f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("Class distribution summary:")
display(pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pcts.round(2)}))


---
## Step 5: Driver-Separated Group Splitting (Preventing Data Leakage)

> ⚠️ **CRITICAL ML BEST PRACTICE**: In driver behavior modeling, a simple random `train_test_split` causes **catastrophic target leakage** because telemetry from the same driver trip appears in both train and test partitions. The model memorizes individual driving quirks rather than generalizing.
>
> We enforce **Strict Driver Separation (Group Partitioning)** where no driver ID (`userid`) appears in both sets.


In [ ]:
# Verify or perform driver-separated train/test partition
if "split" in df.columns:
    train_df = df[df["split"] == "train"].copy()
    test_df = df[df["split"] == "test"].copy()
else:
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df["userid"]))
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()
    train_df["split"] = "train"
    test_df["split"] = "test"

train_drivers = set(train_df["userid"].unique())
test_drivers = set(test_df["userid"].unique())
leakage = train_drivers.intersection(test_drivers)

print(f"📊 Training rows: {len(train_df):,} | Unique drivers: {len(train_drivers):,}")
print(f"📊 Testing rows:  {len(test_df):,} | Unique drivers: {len(test_drivers):,}")
print(f"🔒 Driver Overlap / Data Leakage: {len(leakage)} drivers (MUST BE 0)")

assert len(leakage) == 0, "Data leakage detected! Drivers must be strictly disjoint."
print("✅ Leakage audit passed: Zero driver overlap confirmed.")


---
## Step 6: Baseline Machine Learning Pipeline (Random Forest vs. Naive Baseline)

We train a **Random Forest Classifier** with median imputation and class-weight balancing.
Because this is an imbalanced multi-class problem, we evaluate:
- **Macro-F1**: Equal weight to all 9 classes (rare hazardous states matter as much as normal driving).
- **Balanced Accuracy**: Macro-average of recall across all classes.
- **Confusion Matrix**: Identifies which behaviors (e.g., drowsiness vs. phone use) confuse the model.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, confusion_matrix, accuracy_score
import time

FEATURES = [
    "speed", "gforce", "acc_X", "acc_Y", "acc_Z",
    "accelerometer_data_X", "accelerometer_data_Y", "accelerometer_data_Z",
    "gyroscope_data_X", "gyroscope_data_Y", "gyroscope_data_Z",
    "perclos", "mouth_open_ratio"
]
TARGET = "dangerousstate"

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test = test_df[FEATURES]
y_test = test_df[TARGET]

# 1. Majority-Class Dummy Baseline
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

# 2. Random Forest Baseline Pipeline
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=100,
        max_depth=16,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    ))
])

print("⏳ Training Random Forest baseline on unseen-driver split...")
t0 = time.time()
rf_pipeline.fit(X_train, y_train)
train_time = time.time() - t0
rf_preds = rf_pipeline.predict(X_test)
print(f"✅ Training completed in {train_time:.2f} seconds.")

# Compare metrics
metrics_data = {
    "Model": ["Majority Baseline", "Random Forest (Baseline)"],
    "Accuracy": [accuracy_score(y_test, dummy_preds), accuracy_score(y_test, rf_preds)],
    "Balanced Accuracy": [balanced_accuracy_score(y_test, dummy_preds), balanced_accuracy_score(y_test, rf_preds)],
    "Macro F1": [f1_score(y_test, dummy_preds, average="macro", zero_division=0), f1_score(y_test, rf_preds, average="macro", zero_division=0)],
    "Weighted F1": [f1_score(y_test, dummy_preds, average="weighted", zero_division=0), f1_score(y_test, rf_preds, average="weighted", zero_division=0)]
}
display(pd.DataFrame(metrics_data).round(4))


### Detailed Classification Report & Confusion Matrix
Inspect per-class precision, recall, and common error modes.


In [ ]:
# Generate classification report
labels = sorted(y_test.unique())
print("📋 Classification Report (Unseen Test Drivers):")
print(classification_report(y_test, rf_preds, digits=4, zero_division=0))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, rf_preds, labels=labels, normalize="true")
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Normalized Confusion Matrix (Random Forest on Held-Out Drivers)", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Class")
plt.ylabel("Ground Truth Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


---
## Step 7: Day 4 Feature Ablation Study

Does combining inertial sensors with driver state proxies (`perclos`, `mouth_open_ratio`) improve classification?
We benchmark:
1. **Motion Features Only** (`speed`, `gforce`, `acc_*`, `gyro_*`)
2. **State Features Only** (`perclos`, `mouth_open_ratio`)
3. **Combined Features** (All 13 features)
Across **Random Forest** and **ExtraTreesClassifier**.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

MOTION_FEATURES = [
    "speed", "gforce", "acc_X", "acc_Y", "acc_Z",
    "accelerometer_data_X", "accelerometer_data_Y", "accelerometer_data_Z",
    "gyroscope_data_X", "gyroscope_data_Y", "gyroscope_data_Z"
]
STATE_FEATURES = ["perclos", "mouth_open_ratio"]
COMBINED_FEATURES = MOTION_FEATURES + STATE_FEATURES

feature_sets = {
    "Motion Only": MOTION_FEATURES,
    "State Only": STATE_FEATURES,
    "Combined": COMBINED_FEATURES
}

models = {
    "Random Forest": RandomForestClassifier(n_estimators=60, max_depth=16, class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=60, max_depth=16, class_weight="balanced", n_jobs=-1, random_state=42)
}

ablation_results = []

print("Running feature ablation study across model families...")
for feat_name, feat_cols in feature_sets.items():
    X_tr = train_df[feat_cols]
    X_te = test_df[feat_cols]
    
    for model_name, clf in models.items():
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", clf)
        ])
        t_start = time.time()
        pipe.fit(X_tr, y_train)
        elapsed = time.time() - t_start
        preds = pipe.predict(X_te)
        
        ablation_results.append({
            "Model": model_name,
            "Feature Group": feat_name,
            "Feature Count": len(feat_cols),
            "Accuracy": accuracy_score(y_test, preds),
            "Balanced Acc": balanced_accuracy_score(y_test, preds),
            "Macro F1": f1_score(y_test, preds, average="macro", zero_division=0),
            "Train Time (s)": round(elapsed, 2)
        })

ablation_df = pd.DataFrame(ablation_results).sort_values(by="Macro F1", ascending=False)
display(ablation_df.round(4))

# Visualize Ablation Comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=ablation_df, x="Feature Group", y="Macro F1", hue="Model", palette="Set2")
plt.title("Day 4 Feature Ablation: Macro-F1 across Feature Subsets", fontsize=12, fontweight="bold")
plt.ylabel("Macro-F1 (Higher is Better)")
plt.ylim(0, max(ablation_df["Macro F1"]) * 1.25)
for p in plt.gca().patches:
    h = p.get_height()
    if h > 0:
        plt.gca().annotate(f"{h:.3f}", (p.get_x() + p.get_width() / 2., h),
                           ha='center', va='bottom', fontsize=9, xytext=(0, 3),
                           textcoords='offset points')
plt.tight_layout()
plt.show()


---
## Step 8: Day 5 Computer Vision — Fatigue & Drowsiness Analysis (MediaPipe)

We use the Google MediaPipe Face Landmarker to extract 468 facial mesh keypoints in real time:
- **Eye Aspect Ratio (EAR)**: Computes eye openness to detect micro-sleep episodes and prolonged closures.
- **Mouth Aspect Ratio (MAR)**: Computes lip distance relative to mouth width to detect yawning.


In [ ]:
import cv2
import math
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Geometry helpers for Facial Landmarks
def distance_2d(p1, p2):
    return math.hypot(p1.x - p2.x, p1.y - p2.y)

def calculate_ear(landmarks, eye_indices):
    # eye_indices: [p1, p2, p3, p4, p5, p6]
    vertical_1 = distance_2d(landmarks[eye_indices[1]], landmarks[eye_indices[5]])
    vertical_2 = distance_2d(landmarks[eye_indices[2]], landmarks[eye_indices[4]])
    horizontal = distance_2d(landmarks[eye_indices[0]], landmarks[eye_indices[3]])
    return (vertical_1 + vertical_2) / (2.0 * horizontal + 1e-9)

def calculate_mar(landmarks):
    # Lip top/bottom: 13, 14 | Lip corners: 78, 308
    lip_height = distance_2d(landmarks[13], landmarks[14])
    lip_width = distance_2d(landmarks[78], landmarks[308])
    return lip_height / (lip_width + 1e-9)

# Landmark point indices for eyes
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

print("✅ Facial geometric metrics (EAR / MAR) defined successfully!")


### Process Video Demo with Face Landmarker
Run the landmarker over `data/demo/driving_demo.mp4` (or any uploaded driver video) and log drowsiness/yawning events.


In [ ]:
VIDEO_PATH = Path("data/demo/driving_demo.mp4")
if not VIDEO_PATH.exists():
    # If no sample video is present, search data/demo for any .mp4 file
    videos = list(Path("data/demo").glob("*.mp4"))
    if videos:
        VIDEO_PATH = videos[0]

print(f"Target Video: {VIDEO_PATH}")

if VIDEO_PATH.exists() and TASK_PATH.exists():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Total video frames: {total_frames:,} ({total_frames / fps:.1f}s at {fps:.1f} fps)")

    options = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(TASK_PATH)),
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
    )

    timestamps = []
    ear_values = []
    mar_values = []
    step = max(1, int(round(fps / 5))) # sample ~5 fps for fast analysis

    frame_idx = 0
    with vision.FaceLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret or frame_idx > 300: # process first 300 sampled frames for demonstration
                break
            frame_idx += 1
            if frame_idx % step != 0:
                continue

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            ts_ms = int((frame_idx / fps) * 1000)
            result = landmarker.detect_for_video(mp_image, ts_ms)

            if result.face_landmarks:
                pts = result.face_landmarks[0]
                left_ear = calculate_ear(pts, LEFT_EYE)
                right_ear = calculate_ear(pts, RIGHT_EYE)
                avg_ear = (left_ear + right_ear) / 2.0
                mar = calculate_mar(pts)

                timestamps.append(frame_idx / fps)
                ear_values.append(avg_ear)
                mar_values.append(mar)

    cap.release()
    print(f"Processed {len(timestamps)} sampled frames.")

    # Plot EAR & MAR telemetry
    fig, ax1 = plt.subplots(figsize=(12, 4))
    ax2 = ax1.twinx()

    ax1.plot(timestamps, ear_values, color="dodgerblue", label="Eye Aspect Ratio (EAR)")
    ax1.axhline(0.20, color="blue", linestyle="--", alpha=0.7, label="Drowsiness Threshold (EAR < 0.20)")
    ax1.set_xlabel("Time (seconds)")
    ax1.set_ylabel("EAR (Eye Openness)", color="dodgerblue")

    ax2.plot(timestamps, mar_values, color="darkorange", label="Mouth Aspect Ratio (MAR)")
    ax2.axhline(0.55, color="red", linestyle="--", alpha=0.7, label="Yawning Threshold (MAR > 0.55)")
    ax2.set_ylabel("MAR (Mouth Opening)", color="darkorange")

    plt.title("Driver Fatigue Monitoring (EAR & MAR Temporal Timeline)", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Video not available. Upload driving_demo.mp4 to data/demo/ to run this cell.")


---
## Step 9: Day 5 Computer Vision — YOLO Distracted Driving Detection

We utilize YOLO (Ultralytics) to identify objects indicating distracted driving habits:
- `cell phone` / `smartphone` (using phone while driving)
- `cigarette` / `smoking`
- `cup` / `bottle` (eating / drinking)


In [ ]:
from ultralytics import YOLO

# Load lightweight YOLO model (automatically downloads pretrained weights)
yolo_model = YOLO("yolov8n.pt")
print("✅ Loaded YOLO model:", yolo_model.names)

# Define driving distraction classes
DISTRACTION_CLASSES = ["cell phone", "bottle", "cup", "cigarette"]
print(f"Tracking distraction classes: {DISTRACTION_CLASSES}")


---
## Step 10: Multimodal Journey-Level Risk Scoring

We synthesize telemetry events (high g-force, sudden speed drops, excessive acceleration) and computer vision events (eye closure duration, yawning rate, phone distraction) into a **Journey Safety Score** (0 to 100):
- **100 - 85**: Safe & Attentive Driving
- **84 - 70**: Moderate Risk (Minor distraction / mild fatigue detected)
- **< 70**: High Risk (Action required: frequent eyes-off-road, prolonged yawning, or aggressive maneuvers)


In [ ]:
def calculate_trip_safety_score(drowsy_events_cnt, yawn_cnt, phone_cnt, harsh_braking_cnt, duration_min=10):
    """
    Computes a normalized safety index (0-100) based on weighted infraction penalties.
    """
    base_score = 100.0
    # Penalty weights per infraction
    penalties = {
        "drowsy_episode": drowsy_events_cnt * 12.0,
        "yawn_episode": yawn_cnt * 4.0,
        "phone_usage": phone_cnt * 15.0,
        "harsh_braking": harsh_braking_cnt * 8.0
    }
    total_penalty = sum(penalties.values())
    final_score = max(0.0, min(100.0, base_score - total_penalty))
    
    if final_score >= 85:
        category = "🟢 Safe & Attentive"
    elif final_score >= 70:
        category = "🟡 Moderate Caution"
    else:
        category = "🔴 High Risk / Intervention Recommended"
        
    return {
        "Trip Safety Score": round(final_score, 1),
        "Safety Rating": category,
        "Drowsiness Incidents": drowsy_events_cnt,
        "Yawning Episodes": yawn_cnt,
        "Phone Distraction Events": phone_cnt,
        "Harsh Kinematic Events": harsh_braking_cnt,
        "Total Deductions": round(total_penalty, 1)
    }

# Example Trip Evaluation
sample_report = calculate_trip_safety_score(
    drowsy_events_cnt=2,
    yawn_cnt=3,
    phone_cnt=1,
    harsh_braking_cnt=1,
    duration_min=15
)

print("="*45)
print("       🚗 TRIP SAFETY SUMMARY REPORT       ")
print("="*45)
for k, v in sample_report.items():
    print(f" {k:<25} : {v}")
print("="*45)


---
## Step 11: Launch the Interactive Streamlit Dashboard in Colab

To run the interactive UI (`day5_app.py`) directly from Google Colab, we can expose the local Streamlit port using **LocalTunnel**.
Run the cell below, click on the generated URL, and paste the endpoint IP.


In [ ]:
# Run Streamlit app via localtunnel (Optional UI Demonstration)
import subprocess

# Check if day5_app.py exists
if Path("day5_app.py").exists():
    print("Launching Streamlit app in background on port 8501...")
    process = subprocess.Popen(["streamlit", "run", "day5_app.py", "--server.port", "8501", "--server.headless", "true"])
    
    # Expose port with localtunnel
    !npx -y localtunnel --port 8501 &
    
    # Retrieve public IP to unlock localtunnel
    import urllib.request
    try:
        public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        print(f"🔑 Your LocalTunnel Password / Endpoint IP: {public_ip}")
        print("👉 Click the localtunnel URL above and paste this IP when prompted.")
    except Exception as e:
        print("Note: If running locally, simply visit http://localhost:8501")
else:
    print("day5_app.py not found in current directory.")


---
## Step 12: Project Conclusions, Findings & Responsible AI Disclosures

### 📋 Data Analysis & Modeling Key Findings:
1. **Zero-Leakage Group Partitioning**: Strict driver separation by `userid` prevents optimistic overfitting. Random row-level splitting inflates accuracy, whereas group splitting tests genuine real-world generalization to unseen drivers.
2. **Severe Class Imbalance**: Common classes ("Normal", "No attention") dominate the training space. Balanced subsampling and class-weighted cost functions were essential to prevent the model from ignoring critical hazardous classes like Drowsiness or Phone usage.
3. **Feature Synergy**: Feature ablation confirmed that combining inertial motion features with facial state proxies improves macro-F1 over motion-only or state-only configurations.
4. **Multimodal Edge Fusion**: Smartphone telemetry detects abrupt vehicle dynamics (swerving, harsh braking), while MediaPipe/YOLO vision detects driver internal state (microsleep, eye closure, phone usage), creating a holistic safety evaluation.

### ⚠️ Research & Ethical Disclosures:
- This system is an **academic research prototype**, not a certified automotive safety device.
- Video analysis and eye-tracking require explicit driver consent and strict on-device processing to respect user privacy.
- Raw DriverSVT data is protected by dataset licensing and should not be redistributed publicly.
